# Scrape Upcoming Premier League Fixtures

This notebook scrapes upcoming fixtures from FBref for the 2025-2026 Premier League season.

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from datetime import datetime
import time

## 1. Get Current Season Fixtures Page

In [ ]:
# Premier League Stats page
standings_url = "https://fbref.com/en/comps/9/Premier-League-Stats"
data = requests.get(standings_url)
soup = BeautifulSoup(data.text, 'html.parser')
print("✓ Fetched Premier League page")

## 2. Navigate to Fixtures & Results Page

In [ ]:
# Find the "Scores & Fixtures" link
fixtures_link = None
for link in soup.find_all('a'):
    if 'schedule' in link.get('href', ''):
        fixtures_link = link.get('href')
        break

if fixtures_link:
    fixtures_url = f"https://fbref.com{fixtures_link}"
    print(f"✓ Found fixtures page: {fixtures_url}")
else:
    print("✗ Could not find fixtures link")

In [ ]:
# Fetch fixtures page
time.sleep(3)  # Be nice to the server
fixtures_data = requests.get(fixtures_url)
fixtures_soup = BeautifulSoup(fixtures_data.text, 'html.parser')
print("✓ Fetched fixtures page")

## 3. Extract Fixtures Table

In [ ]:
# Get all fixtures from the table
fixtures_df = pd.read_html(fixtures_data.text, match="Scores & Fixtures")[0]
print(f"✓ Found {len(fixtures_df)} total matches (past and future)")
fixtures_df.head(10)

## 4. Filter for Upcoming Matches

In [ ]:
# Convert Date column to datetime
fixtures_df['Date'] = pd.to_datetime(fixtures_df['Date'], errors='coerce')

# Get today's date
today = datetime.now()

# Filter for future matches (no score yet)
# Check if 'Score' column exists, otherwise use different method
if 'Score' in fixtures_df.columns:
    upcoming = fixtures_df[fixtures_df['Score'].isna()].copy()
else:
    # If no Score column, filter by date
    upcoming = fixtures_df[fixtures_df['Date'] >= today].copy()

print(f"✓ Found {len(upcoming)} upcoming matches")
upcoming.head(20)

## 5. Clean and Format Fixtures Data

In [ ]:
# Select and rename relevant columns
fixtures_clean = upcoming[['Date', 'Time', 'Home', 'Away', 'Venue']].copy()

# Rename columns for consistency
fixtures_clean.columns = ['match_date', 'match_time', 'home_team', 'away_team', 'venue']

# Sort by date
fixtures_clean = fixtures_clean.sort_values('match_date').reset_index(drop=True)

print(f"✓ Cleaned {len(fixtures_clean)} upcoming fixtures")
fixtures_clean.head(20)

## 6. Export Fixtures

In [ ]:
# Save to CSV
fixtures_clean.to_csv('upcoming_fixtures.csv', index=False)
print(f"✓ Saved {len(fixtures_clean)} fixtures to upcoming_fixtures.csv")

In [ ]:
# Also save to backend data directory for the API
import shutil
from pathlib import Path

data_dir = Path('../data')
data_dir.mkdir(exist_ok=True)

shutil.copy('upcoming_fixtures.csv', data_dir / 'upcoming_fixtures.csv')
print(f"✓ Copied fixtures to {data_dir / 'upcoming_fixtures.csv'}")

## 7. Preview Next 10 Fixtures

In [ ]:
print("\n=== NEXT 10 PREMIER LEAGUE FIXTURES ===\n")
for idx, row in fixtures_clean.head(10).iterrows():
    date_str = row['match_date'].strftime('%a, %b %d, %Y')
    time_str = row['match_time'] if pd.notna(row['match_time']) else 'TBD'
    print(f"{date_str} at {time_str}")
    print(f"  {row['home_team']} vs {row['away_team']}")
    print(f"  Venue: {row['venue']}")
    print()

## 8. Summary Statistics

In [ ]:
print("\n=== FIXTURES SUMMARY ===\n")
print(f"Total upcoming fixtures: {len(fixtures_clean)}")
print(f"\nFirst fixture: {fixtures_clean['match_date'].min().strftime('%B %d, %Y')}")
print(f"Last fixture: {fixtures_clean['match_date'].max().strftime('%B %d, %Y')}")
print(f"\nTeams in upcoming fixtures:")
all_teams = set(fixtures_clean['home_team'].unique()) | set(fixtures_clean['away_team'].unique())
print(f"  {len(all_teams)} teams: {sorted(all_teams)}")